<a href="https://www.kaggle.com/code/asgx1224/hftbacktest-claude?scriptVersionId=356791961" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# 1. 安装 hftbacktest。Kaggle 每次重开会话都要重新运行这一行。
#    -q 表示安静模式，只显示错误，不输出安装过程的大量日志。
!pip install -q hftbacktest

# 2. 确认安装成功，打印版本号。
#    importlib.metadata 是 Python 标准库，用来读取已安装包的版本。
from importlib.metadata import version
print("hftbacktest 版本:", version("hftbacktest"))

# 3. 打开 npz 文件。
#    .npz 本质是多个 numpy 数组打包在一起的压缩文件，
#    np.load 只是打开它，此时还没有把全部数据读进内存。
import numpy as np
path = "/kaggle/input/datasets/asgx1224/hftbacktest/BTCUSDT_2023-04-01.npz"
f = np.load(path)

# 4. 列出文件里有哪些数组（键名）。
print("包含的键:", f.files)

# 5. 逐个查看每个数组的形状（行数、列数）和数据类型。
#    f[k] 这时才会真正读取该数组，一次只读一个，避免内存爆掉。
for k in f.files:
    arr = f[k]
    print(k, "形状:", arr.shape, "类型:", arr.dtype)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 58.3 MB/s eta 0:00:00
hftbacktest 版本: 2.4.4
包含的键: ['data']
data 形状: (60943533,) 类型: [('ev', '<u8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')]


In [2]:
import pandas as pd
from numba import njit
data = f["data"]   # 取出数组。这一步不复制数据，只是拿到引用。

# 直接打印前 5 行，每一行是一个事件，显示全部字段。
print(data[:5])

# 把交易所时间戳（纳秒整数）转换成日期时间，便于阅读。
# unit="ns" 告诉 pandas 输入的整数单位是纳秒。
print(pd.to_datetime(data["exch_ts"][:5], unit="ns"))

# 只看 ev 字段的原始值，十六进制显示，方便之后对照位标志。
print([hex(x) for x in data["ev"][:5]])

[(3489660930, 1680307200158000000, 1680307200165963000, 28454.8, 0.009, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173448000, 28454.8, 0.027, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173512000, 28454.8, 0.005, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200174315000, 28454.8, 0.022, 0, 0, 0.)
 (3489660930, 1680307200183000000, 1680307200190901000, 28454.8, 0.004, 0, 0, 0.)]
DatetimeIndex(['2023-04-01 00:00:00.158000', '2023-04-01 00:00:00.165000',
               '2023-04-01 00:00:00.165000', '2023-04-01 00:00:00.165000',
               '2023-04-01 00:00:00.183000'],
              dtype='datetime64[ns]', freq=None)
['0xd0000002', '0xd0000002', '0xd0000002', '0xd0000002', '0xd0000002']


In [3]:
import numpy as np
from hftbacktest.data import validate_event_order

# ---- 你的实测值（P50，单位纳秒）----
DEPTH_FEED_NS = 677_000       # 行情延迟 Depth 0.677 ms
TRADE_FEED_NS = 1_398_000     # 行情延迟 Trade 1.398 ms
ENTRY_NS      = 490_390       # 下单延迟 0.49039 ms
RESP_NS       = 494_130       # 响应延迟 0.49413 ms

# 全天 6094 万行，处理峰值内存约 20GB，Kaggle 可能撑不住。
# 先只取前 1 小时做实验；设为 None 则用全天。
LIMIT_HOURS = 1

EXCH  = np.uint64(0x80000000)   # EXCH_EVENT 标志（单元格 5 已验证）
LOCAL = np.uint64(0x40000000)   # LOCAL_EVENT 标志（由 0xd0000002 推出）
TYPE_MASK = np.uint64(0xFF)

if LIMIT_HOURS is None:
    d = data.copy()
else:
    t_end = int(data["exch_ts"][0]) + int(LIMIT_HOURS * 3600 * 1e9)
    d = data[data["exch_ts"] <= t_end].copy()

# 1) 行情延迟：local_ts = exch_ts + 实测延迟（成交 1.398ms，其余 0.677ms）
is_trade = (d["ev"] & TYPE_MASK) == 2
d["local_ts"] = d["exch_ts"] + np.where(is_trade, TRADE_FEED_NS, DEPTH_FEED_NS)

# 2) 拆成"交易所流"和"本地流"。
#    EXCH 流按 exch_ts 排序，LOCAL 流按 local_ts 排序，两条流互不影响。
e = d[(d["ev"] & EXCH) != 0].copy()
e["ev"] = e["ev"] & ~LOCAL
l = d[(d["ev"] & LOCAL) != 0].copy()
l["ev"] = l["ev"] & ~EXCH
del d

merged = np.concatenate([e, l])
key = np.concatenate([e["exch_ts"], l["local_ts"]])
n_e, n_l = len(e), len(l)
del e, l

order = np.argsort(key, kind="stable")        # 稳定排序，同一时刻保持原有先后
data2 = merged[order]
del merged, key, order

# 3) 官方校验：两条流都有序才会通过
validate_event_order(data2)
print("校验通过 | 行数:", len(data2), "| EXCH 行:", n_e, "| LOCAL 行:", n_l)

tr = (data2["ev"] & TYPE_MASK) == 2
loc = (data2["ev"] & LOCAL) != 0
print("LOCAL 流行情延迟(ms)  Depth:", np.median((data2["local_ts"] - data2["exch_ts"])[loc & ~tr]) / 1e6,
      " Trade:", np.median((data2["local_ts"] - data2["exch_ts"])[loc & tr]) / 1e6)

校验通过 | 行数: 5726828 | EXCH 行: 2863414 | LOCAL 行: 2863414
LOCAL 流行情延迟(ms)  Depth: 0.677  Trade: 1.398


In [4]:
from hftbacktest import FILLED, GTC, LIMIT, NEW,BacktestAsset,HashMapMarketDepthBacktest

asset_real = (
    BacktestAsset()
    .data([data2])                              # 已改写 local_ts 的数组
    .linear_asset(1.0)
    .tick_size(0.1)
    .lot_size(0.001)
    .constant_latency(ENTRY_NS, RESP_NS)        # 下单 / 响应延迟
    .risk_adverse_queue_model()                 # 队列模型显式写出
    .no_partial_fill_exchange()
)

@njit
def rest_at_best_bid_v2(hbt, step_ns, max_seconds):
    order_id = 1
    submitted = False
    last_status = -1
    submit_ts = 0
    while hbt.elapse(step_ns) == 0:
        depth = hbt.depth(0)
        if not np.isfinite(depth.best_bid):
            continue
        if not submitted:
            price = depth.best_bid
            hbt.submit_buy_order(0, order_id, price, 0.01, GTC, LIMIT, False)
            submitted = True
            submit_ts = hbt.current_timestamp
            print("提交", submit_ts, "价格", np.round(price, 1))
        else:
            order = hbt.orders(0).get(order_id)
            if order.status != last_status:
                last_status = order.status
                print("状态", order.status,
                      "时间", hbt.current_timestamp,
                      "距提交(ms)", (hbt.current_timestamp - submit_ts) / 1e6,
                      "最优买", np.round(depth.best_bid, 1),
                      "最优卖", np.round(depth.best_ask, 1),
                      "持仓", hbt.position(0))
            if order.status == FILLED:
                return True
            if hbt.current_timestamp - submit_ts >= max_seconds * 1e9:
                return False
    return False


# 步长 1ms：1 秒的步长会把 0.5ms 的延迟完全淹没
hbt = HashMapMarketDepthBacktest([asset_real])
print("【真实延迟】是否成交:", rest_at_best_bid_v2(hbt, 1e6, 1800))
hbt.close()

/tmp/ipykernel_16/1072854901.py:9: DeprecationWarning: constant_latency() is deprecated; use constant_order_latency().
  .constant_latency(ENTRY_NS, RESP_NS)        # 下单 / 响应延迟


提交 1680307200198000000 价格 28454.8
状态 1 时间 1680307200199000000 距提交(ms) 1.0 最优买 28454.8 最优卖 28454.9 持仓 0.0
状态 3 时间 1680307204991000000 距提交(ms) 4793.0 最优买 28454.8 最优卖 28454.9 持仓 0.01
【真实延迟】是否成交: True


0

In [5]:
#练习
@njit
def myfirst_test(hbt):
    count=1
    
    while hbt.elapse(60*1e9)==0:
        depth=hbt.depth(0)
        if count>3:
            continue
        else:
            print("当前时间：",hbt.current_timestamp)

            tick_size=depth.tick_size
            for i in range(depth.best_ask_tick,depth.best_ask_tick+3):
                print(round(i * tick_size,2),"@",depth.ask_qty_at_tick(i) )

            

        
        count+=1
hbt = HashMapMarketDepthBacktest([asset_real])
myfirst_test(hbt)
hbt.close()

当前时间： 1680307260158000000
28474.8 @ 3.304
28474.9 @ 0.002
28475.0 @ 0.059
当前时间： 1680307320158000000
28455.2 @ 0.592
28455.3 @ 0.029
28455.4 @ 0.028
当前时间： 1680307380158000000
28454.0 @ 11.017
28454.1 @ 0.001
28454.2 @ 0.28


0